# Initial Experiments — Acórdãos with LLMs

This notebook contains the initial experiments and prototyping of the proposed tasks:
- Structured technical brief
- Classification
- Entity extraction
- Contextualized QA implementation

## Setup

In [1]:
import os

if not os.path.exists("docs"):
    os.chdir("..")
    assert os.path.exists("docs")

In [2]:
from dotenv import load_dotenv
from pprint import pprint
from utils.llm_adapter import LLMAdapter
from utils.task_handlers import (
    summarize_acordao,
    classify_acordao,
    extract_entities,
    qa_acordao,
)
from ocr.file_handler import extract_text 
import json

# Load environment variables from .env file
load_dotenv()

True

## Load example data

In [ ]:
with open("docs/acordao_examples/apelacao_latrocinio.txt", "r") as f:
    latrocinio = f.read()

with open("docs/acordao_examples/maria_da_penha.txt", "r") as f:
    maria_da_penha = f.read()

with open("docs/acordao_examples/20220531.json", "r", encoding="utf-8") as f:
    # acordãos segunda turma
    acordaos: list[dict] = json.load(f)

# using OCR
habeascorpus = extract_text("docs/acordao_examples/habeascorpus.pdf")

In [7]:
print(habeascorpus[:500])

Página 1:
Banco de Jurisprudéncia - PJe

TRIBUNAL DE JUSTICA DA PARAIBA

0802708-25.2025.8.15.0000

Classe: HABEAS CORPUS CRIMINAL

Orgao Julgador: Camara Criminal

Relator: Gabinete 12 - Des. Carlos Martins Beltrdo Filho

Origem: TJPB - Tribunal Pleno, Camaras e Secdes Especializadas
Tipo do documento: Acérdao

Data de juntada: 25/04/2025

Ementa EMENTA SEM FORMATACAO

PODER JUDICIARIO
Tribunal de Justica da Paraiba
Gabinete 12 - Desembargador CARLOS Martins BELTRAO Filho

HABEAS CORPUS CRIMINA


## 1. Structured Summary Generation

The goal here is, based on an unstructured Acordao, to generate a minimally structured summary with two section: 
1. **Ementa**
2. **Decisão**

All judgments will contain these two informations.

---
Below I run the same Acórdão with two different models:
- llama3.2 (3b parameters)
  - This model is so small that I can run it on my humble 4gb VRAM graphic card
- GPT-4.1-nano
  - Currently, OpenAI's most cost-effective model

In the results, the difference in the quality of the information is clear (gpt wins), but it is also clear that the structure remains the same, with the two headers defined.

In [4]:
llm_from_local = LLMAdapter(provider="ollama", model_name="llama3.2:3b", temperature=0.3)
llm_from_api = LLMAdapter(provider="openai", model_name="gpt-4.1-nano", api_key=os.getenv("OPENAI_API_KEY"))

In [ ]:
# Local

summary_local = summarize_acordao(llm=llm_from_local, acordao=maria_da_penha)
print(summary_local)

**Ementa Resumida:**

O acórdão do Tribunal de Justiça da Paraíba nega a ordem de habeas corpus preventivo impetrada pelo paciente Alexsandro Barbosa Guedes, que alega falta de fundamentação na decisão que o decretou sua prisão preventiva. A decisão do juiz foi baseada em informações prestadas pela vítima, que foram consideradas inverídicas pelo paciente. O acórdão mantém a prisão preventiva do paciente, argumentando que a ameaça cometida por ex-namorado configura violência doméstica e que o paciente possui condições pessoais favoráveis.

**Decisão Resumida:**

A decisão do Tribunal de Justiça da Paraíba nega a ordem de habeas corpus preventivo impetrada pelo paciente Alexsandro Barbosa Guedes, baseando-se na decisão do juiz que o decretou sua prisão preventiva. A Corte considera que a ameaça cometida por ex-namorado configura violência doméstica e que o paciente possui condições pessoais favoráveis. Além disso, a decisão do juiz foi baseada em informações prestadas pela vítima, que fo

In [ ]:
# API

summary_api = summarize_acordao(llm=llm_from_api, acordao=maria_da_penha)
print(summary_api)

**Ementa Resumida:**  
Habeas corpus impetrado contra decisão que decretou a prisão preventiva com base na Lei Maria da Penha, fundamentada na garantia da ordem pública e na necessidade de proteção à vítima, diante de fatos concretos de ameaças e violência doméstica. A alegação de ausência de fundamentação e de fatos inverídicos foi considerada questão de mérito, não apreciável na via do habeas corpus. A jurisprudência do STJ reconhece a aplicação da Lei nº 11.340/06 mesmo após o término do relacionamento, quando há ameaça ou violência de ex-namorado. A decisão que decretou a prisão apresentou fundamentos suficientes, demonstrando a necessidade da medida para resguardar a ordem pública e a integridade da vítima.  

**Decisão Resumida:**  
A ordem de habeas corpus foi indeferida, pois a decisão que decretou a prisão preventiva do paciente estava devidamente fundamentada, atendendo aos requisitos legais e evidenciando a necessidade da medida para garantir a ordem pública e a proteção da 

## 2. Classification

The goal here is to classify the Acórdão with pre-defined classes. \
For this task, I thought of two types of classification:

1. **Categoria:** A [multilabel](https://en.wikipedia.org/wiki/Multi-label_classification) data that can take the following values: `administrativo`, `cível`, `penal`, `trabalhista`, `tributário` and `outro`.
2. **Decisao:** The final decision of the judge. It can assume only one of the following values: `favorável`, `desfavorável` or `neutro`

Here they are forced by their own frameworks (langchain or ollama) to respond in a structured way

In [ ]:
# Local

classification_local = classify_acordao(llm=llm_from_local, acordao=maria_da_penha)
print(classification_local)

categoria_do_processo=['administrativo', 'cível'] decisao='desfavorável'


In [ ]:
# API

classification_api = classify_acordao(llm=llm_from_api, acordao=maria_da_penha)
print(classification_api)

categoria_do_processo=['penal'] decisao='desfavorável'


## 3. Entity Extraction

The objective here is to extract entities related to the Acórdão. The following is the list of entities to be extracted:

1. Classe do processo: Ex.: Recurso Especial, Agravo de Instrumento, etc.
2. Numero identificador do processo.
3. Entidade Julgadora: Ex.: Primeira Turma, Camara Criminal, etc.
4. Relator.
5. Lista de Pessoas Citadas: List of persons/entities that appear in the judgment and their respective positions/functions.
6. Data de publicação.
7. Data de decisão.
8. Lista de Fundamentos Jurídicos.


As with the classification task, the output is also structured and validated.

> Note: For this task, I used the one-shot prompting technique, in which I simulate a message from the user (the Acordão) and a response from the assistant (the extracted entities). The structure can be found in the `extraction_chat_messages(text)` function in `utils/prompt_helpers.py`

In [ ]:
# Local

entities_local = extract_entities(llm=llm_from_local, acordao=maria_da_penha)
pprint(entities_local.model_dump())

{'data_decisao': '31/07/2015',
 'data_publicacao': None,
 'descricao_classe_processo': 'HABEAS CORPUS',
 'entidade_julgadora': 'Câmara Criminal',
 'fundamentos_juridicos': ['Lei nº 11.340/06',
                           'artigos relacionados à Lei Maria da Penha',
                           'Precedentes do STJ: HC n. 126.912/SP, Ministra '
                           'Maria Thereza de Assis Moura, Dje 12/4/2010'],
 'numero_processo': '0800888-20.2015.8.15.0000',
 'pessoas_citadas': [{'cargo': 'Impetrante/Advogado',
                      'nome': 'Mario Felix de Menezes'},
                     {'cargo': 'Paciente', 'nome': 'Alexsandro Barbosa Guedes'},
                     {'cargo': 'Autoridade coatora',
                      'nome': 'JUIZ DE DIREITO'}],
 'relator': 'Des. Joás de Brito Pereira Filho'}


In [ ]:
# API

entities_api = extract_entities(llm=llm_from_api, acordao=maria_da_penha)
pprint(entities_api.model_dump())

{'data_decisao': '31/07/2015',
 'data_publicacao': '31/07/2015',
 'descricao_classe_processo': 'HABEAS CORPUS',
 'entidade_julgadora': 'Câmara Criminal',
 'fundamentos_juridicos': ['Lei nº 11.340/06',
                           'artigos relacionados à Lei Maria da Penha',
                           'Precedentes do STJ: HC n. 126.912/SP, Ministra '
                           'Maria Thereza de Assis Moura, Dje 12/4/2010'],
 'numero_processo': '0800888-20.2015.8.15.0000',
 'pessoas_citadas': [{'cargo': 'Impetrante/Advogado',
                      'nome': 'Mario Felix de Menezes'},
                     {'cargo': 'Paciente', 'nome': 'Alexsandro Barbosa Guedes'},
                     {'cargo': 'Autoridade coatora',
                      'nome': 'JUIZ DE DIREITO'}],
 'relator': 'Des. Joás de Brito Pereira Filho'}


## 4. Question Answering

Here one is free to ask anything according to the Acórdão of context.

For this activity, I included the document as part of the system prompt, that is, the first message in the list. Based on this, at each iteration, the model will have the document as a context.

In [15]:
# API

messages = []
while True:
    user_input = input("Digite sua pergunta (ou 'sair' para encerrar): ")
    if user_input.lower() == "sair":
        break

    response = qa_acordao(
        llm=llm_from_api,
        acordao=maria_da_penha,
        question=user_input,
        chat_history=messages,
    )

    print("Eu:", user_input)
    if messages[-1][0] == "assistant":
        print("Assistente:", messages[-1][1])

Eu: houve caso de violência doméstica?
Assistente: Sim, houve caso de violência doméstica. O acórdão menciona que o paciente, Alexsandro Barbosa Guedes, foi denunciado por práticas reiteradas de violência doméstica, incluindo ameaças de morte contra a vítima, além de ter tentado ingressar na residência da ofendida contra sua vontade, mesmo após quase um ano de rompimento do relacionamento. Essas circunstâncias justificaram a decretação da prisão preventiva do paciente, com base na necessidade de resguardar a ordem pública e a integridade física da vítima.
Eu: quem é Alexsandro Barbosa?
Assistente: Alexsandro Barbosa Guedes é o paciente mencionado no acórdão, que foi alvo de habeas corpus impetrado em sua defesa. Ele foi denunciado por práticas de violência doméstica, incluindo ameaças de morte e tentativa de ingressar na residência da vítima contra sua vontade. A decisão judicial tratou de sua prisão preventiva em decorrência dessas circunstâncias.
Eu: o habeas corpus foi aceito?
Assis

In [16]:
# Local

messages = []
while True:
    user_input = input("Digite sua pergunta (ou 'sair' para encerrar): ")
    if user_input.lower() == "sair":
        break

    response = qa_acordao(
        llm=llm_from_local,
        acordao=maria_da_penha,
        question=user_input,
        chat_history=messages,
    )

    print("Eu:", user_input)
    if messages[-1][0] == "assistant":
        print("Assistente:", messages[-1][1])

Eu: houve caso de violência doméstica?
Assistente: Sim, o acórdão menciona que há notícias de reiteradas práticas de violência doméstica contra a vítima, incluindo ameaças de morte e tentativas de ingresso na residência da vítima sem seu consentimento. Além disso, o magistrado considera que a ameaça cometida pelo ex-namorado do paciente configura violência doméstica, ensejando a aplicação da Lei Maria da Penha.

O acórdão destaca que o paciente já não namorava a vítima no momento da suposta ameaça, mas isso não impede a aplicação da Lei Maria da Penha, pois a relação íntima de afeto entre os dois é considerada.
Eu: quem é Alexsandro Barbosa?
Assistente: Não há informações explícitas sobre quem é Alexsandro Barbosa em relação ao caso apresentado no acórdão. Ele é o paciente que impetrou o habeas corpus preventivo, alegando que sua prisão preventiva foi decretaada sem fundamentação adequada e que ele sofre constrangimento ilegal devido à sua situação.

É mencionado que Alexsandro Barbosa

# Other experiments
---

## Tokens x Characters

### How to know if a document is too large to the model context?

If a document is too large, that is, its number of tokens is bigger than the model supports, it will ignore some parts of the document

In this expermient we can see that the number of tokens is approximately 4x the amount of characters.
So, to know the number of tokens we do: 

```python
num_tokens = len(document) / 4
``` 

- llama3.2: 128,000 tokens of context
- gpt-4o: 128,000 tokens of context
- gpt-4.1-nano: 1,047,576 tokens of context

So it seems that we don't need to worry about the size of the context

In [ ]:
import tiktoken

with open("docs/exemplos_acordaos/apelacao_latrocinio.txt", "r") as f:
    acordao = f.read()

open_ai_tokenizer = tiktoken.encoding_for_model("gpt-4o")

tokens = open_ai_tokenizer.encode(acordao)
print("Amount characters:", len(acordao))
print("Amount tokens: ", len(tokens))

print(f"tokens/characters ratio: {len(tokens) / len(acordao):.2f}", )
print("The amount of tokens is approximately 4x the amount of characters.")

Amount characters: 34910
Amount tokens:  8864
tokens/characters ratio: 0.25
The amount of tokens is approximately 4x the amount of characters.
